# Windblade YOLO11n train/validation apparatus
This output-free notebook runs the three frozen class-agnostic seeds. It uses only training and validation data; the held-out split remains sealed. Run cells in order on a Colab GPU runtime with at least 8 GiB VRAM.

In [1]:
from pathlib import Path
import csv
import json
import subprocess
import sys

repository = Path("/content/Bade-defect-recognition")
data_root = Path("/content/windblade_phase11b_data")
drive_root = Path("/content/drive/MyDrive/windblade_phase11b")

print("PYTHON:", sys.version)
print("REPOSITORY EXISTS:", repository.exists())
print("LOCAL DATA EXISTS:", data_root.exists())
print("DRIVE MOUNTED:", Path("/content/drive/MyDrive").exists())
print("DRIVE ROOT EXISTS:", drive_root.exists())

processes = subprocess.run(
    ["ps", "-eo", "pid,etime,args"],
    text=True,
    capture_output=True,
    check=True,
).stdout.splitlines()

training_processes = [
    line for line in processes
    if "scripts/run_phase11b.py" in line and " train " in line
]

print("\nACTIVE TRAINING PROCESSES:")
if training_processes:
    for line in training_processes:
        print(line)
else:
    print("NONE")

for seed in (17, 29, 43):
    run_dir = drive_root / "runs" / f"seed_{seed}"
    state_path = run_dir / "run_state.json"
    results_path = run_dir / "results.csv"

    status = None
    if state_path.is_file():
        status = json.loads(state_path.read_text()).get("status")

    rows = 0
    last_epoch = None
    if results_path.is_file():
        with results_path.open(newline="") as handle:
            results = list(csv.DictReader(handle))
        rows = len(results)
        if results:
            last_epoch = results[-1].get("epoch")

    print(
        f"\nSEED {seed}:",
        {
            "directory": run_dir.exists(),
            "status": status,
            "result_rows": rows,
            "last_epoch": last_epoch,
            "last_checkpoint": (run_dir / "weights" / "last.pt").is_file(),
        },
    )

PYTHON: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
REPOSITORY EXISTS: False
LOCAL DATA EXISTS: False
DRIVE MOUNTED: False
DRIVE ROOT EXISTS: False

ACTIVE TRAINING PROCESSES:
NONE

SEED 17: {'directory': False, 'status': None, 'result_rows': 0, 'last_epoch': None, 'last_checkpoint': False}

SEED 29: {'directory': False, 'status': None, 'result_rows': 0, 'last_epoch': None, 'last_checkpoint': False}

SEED 43: {'directory': False, 'status': None, 'result_rows': 0, 'last_epoch': None, 'last_checkpoint': False}


In [3]:
from google.colab import drive
from pathlib import Path
import csv
import json

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/windblade_phase11b")
archive = drive_root / "inputs" / "WT blade defect dataset.zip"

print("DRIVE ROOT EXISTS:", drive_root.exists())
print("ARCHIVE EXISTS:", archive.is_file())

for seed in (17, 29, 43):
    run_dir = drive_root / "runs" / f"seed_{seed}"
    state_path = run_dir / "run_state.json"
    results_path = run_dir / "results.csv"
    last_path = run_dir / "weights" / "last.pt"

    status = None
    if state_path.is_file():
        status = json.loads(state_path.read_text()).get("status")

    rows = 0
    last_epoch = None
    if results_path.is_file():
        with results_path.open(newline="") as handle:
            results = list(csv.DictReader(handle))
        rows = len(results)
        if results:
            last_epoch = results[-1].get("epoch")

    print(
        f"SEED {seed}:",
        {
            "directory": run_dir.exists(),
            "status": status,
            "result_rows": rows,
            "last_epoch": last_epoch,
            "last_checkpoint": last_path.is_file(),
        },
    )

Mounted at /content/drive
DRIVE ROOT EXISTS: True
ARCHIVE EXISTS: True
SEED 17: {'directory': True, 'status': 'TRAINING_COMMAND_COMPLETED', 'result_rows': 100, 'last_epoch': '100', 'last_checkpoint': True}
SEED 29: {'directory': True, 'status': 'TRAINING_COMMAND_COMPLETED', 'result_rows': 100, 'last_epoch': '100', 'last_checkpoint': True}
SEED 43: {'directory': True, 'status': None, 'result_rows': 38, 'last_epoch': '38', 'last_checkpoint': True}


In [4]:
from pathlib import Path
import subprocess

REPOSITORY_ROOT = "/content/Bade-defect-recognition"
DRIVE_ROOT = "/content/drive/MyDrive/windblade_phase11b"
DATA_ROOT = "/content/windblade_phase11b_data"
EXECUTION_COMMIT = "764dd67b31616fa0165a1fe4058b82eb978e4178"

root = Path(REPOSITORY_ROOT)

assert not root.exists(), (
    f"STOP: {root} already exists; do not overwrite it"
)

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/Stukhori/Bade-defect-recognition.git",
        REPOSITORY_ROOT,
    ],
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", EXECUTION_COMMIT],
    cwd=REPOSITORY_ROOT,
    check=True,
)

head = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

print("HEAD:", head)
print("STATUS:", repr(status))

assert head == EXECUTION_COMMIT
assert status == ""

print("CORRECTED REPOSITORY CHECKOUT: PASS")

HEAD: 764dd67b31616fa0165a1fe4058b82eb978e4178
STATUS: ''
CORRECTED REPOSITORY CHECKOUT: PASS


In [6]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import windblade; "
            "from windblade.detection import phase11b; "
            "print('WINDBLADE:', windblade.__file__); "
            "print('PHASE11B:', phase11b.__file__)"
        ),
    ],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

assert result.returncode == 0, "Fresh-process package import failed"
print("EDITABLE INSTALLATION: PASS")

RETURN CODE: 0
STDOUT:
WINDBLADE: /content/Bade-defect-recognition/src/windblade/__init__.py
PHASE11B: /content/Bade-defect-recognition/src/windblade/detection/phase11b.py

STDERR:

EDITABLE INSTALLATION: PASS


In [7]:
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

for command in ("apparatus-check", "verify-archive", "preflight"):
    result = subprocess.run(
        BASE + [command],
        cwd=REPOSITORY_ROOT,
        text=True,
        capture_output=True,
    )

    print(f"\n========== {command} ==========")
    print("RETURN CODE:", result.returncode)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"{command} failed; stop here")

print("RUNTIME SAFETY CHECKS: PASS")


========== apparatus-check ==========
RETURN CODE: 0
STDOUT:
{
  "boxes": 1065,
  "boxes_by_split": {
    "test": 162,
    "train": 757,
    "validation": 146
  },
  "config_fingerprint": "9f4a20ba4404c9a6072277a504c466a0756143b908e79c7168d2ccf91ff32057",
  "dataset_fingerprint": "ad4ab59c3e3c85c6cf0b85b148177bd6b79d24f372f49bdff0043609e6fefc97",
  "images": 720,
  "images_by_split": {
    "test": 109,
    "train": 510,
    "validation": 101
  },
  "run_count": 3,
  "scientific_output_fingerprint": "3f46cbdc6c7a2e3cf6093ff177dd1948d113fa4c36fa9eb907d7c8621e800461",
  "seeds": [
    17,
    29,
    43
  ],
  "split_fingerprint": "264f8460f203074374c2c098c8fd5d2e55fb7ee1f281a8d505e2dfb0de9a2bc3",
  "status": "PASS"
}

STDERR:


========== verify-archive ==========
RETURN CODE: 0
STDOUT:
{
  "filename": "WT blade defect dataset.zip",
  "sha256": "466452f2a0cfc9ef6ba63ea2a3bbc7ea4262057dd07e4fc9e00eedf5bba305b4",
  "size_bytes": 78958553,
  "status": "PASS"
}

STDERR:


========== preflig

In [8]:
import os
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

print("Recreating temporary train/validation data...")
subprocess.run(
    BASE + ["materialize-trainval"],
    cwd=REPOSITORY_ROOT,
    check=True,
)

print("Resuming seed 43 from its saved checkpoint...")
environment = dict(
    os.environ,
    PYTHONHASHSEED="43",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

subprocess.run(
    BASE + ["train", "--seed", "43"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    check=True,
)

print("SEED 43 TRAINING COMMAND: COMPLETE")

Recreating temporary train/validation data...
Resuming seed 43 from its saved checkpoint...
SEED 43 TRAINING COMMAND: COMPLETE


In [9]:
from pathlib import Path
import csv
import hashlib
import json

run_dir = Path(DRIVE_ROOT) / "runs" / "seed_43"
state = json.loads((run_dir / "run_state.json").read_text())

with (run_dir / "results.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))

last_path = run_dir / "weights" / "last.pt"

digest = hashlib.sha256()
with last_path.open("rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)
observed_sha = digest.hexdigest()

print("STATUS:", state.get("status"))
print("RESULT ROWS:", len(rows))
print("LAST EPOCH:", rows[-1].get("epoch"))
print("BATCH:", state.get("applied_batch_size"))
print("OPTIMIZER:", state.get("applied_optimizer"))
print("AMP:", state.get("applied_amp"))
print("CHECKPOINT HASH MATCH:", state.get("last_checkpoint_sha256") == observed_sha)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["seed"] == 43
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["last_checkpoint_sha256"] == observed_sha
assert len(rows) > 38

print("SEED 43 COMPLETION: PASS")

STATUS: TRAINING_COMMAND_COMPLETED
RESULT ROWS: 100
LAST EPOCH: 100
BATCH: 16
OPTIMIZER: AdamW
AMP: True
CHECKPOINT HASH MATCH: True
SEED 43 COMPLETION: PASS


In [10]:
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

subprocess.run(
    BASE + ["select-validation"],
    cwd=REPOSITORY_ROOT,
    check=True,
)

print("VALIDATION-ONLY SELECTION: COMPLETE")
print(
    "Receipt:",
    f"{REPOSITORY_ROOT}/provenance/phase11b_selection_receipt.json",
)

VALIDATION-ONLY SELECTION: COMPLETE
Receipt: /content/Bade-defect-recognition/provenance/phase11b_selection_receipt.json


In [11]:
from pathlib import Path
import hashlib
import json
import subprocess

from google.colab import files

receipt_path = (
    Path(REPOSITORY_ROOT)
    / "provenance"
    / "phase11b_selection_receipt.json"
)
drive_root = Path(DRIVE_ROOT)
selection_root = drive_root / "selection"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

receipt = json.loads(receipt_path.read_text())

assert receipt["status"] == "FROZEN_BEFORE_TEST"
assert receipt["test_evaluated"] is False
assert receipt["no_post_test_tuning"] is True
assert receipt["seeds"] == [17, 29, 43]
assert len(receipt["checkpoints"]) == 3
assert receipt["configuration"]["sha256"] == (
    "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
)

for checkpoint in receipt["checkpoints"]:
    path = drive_root / checkpoint["drive_relative_path"]
    assert path.is_file()
    assert path.stat().st_size == checkpoint["size_bytes"]
    assert sha256(path) == checkpoint["sha256"]

candidate_path = selection_root / "validation_checkpoint_candidates.json"
threshold_path = selection_root / "validation_threshold_candidates.json"

assert sha256(candidate_path) == receipt["validation_artifacts"][
    "checkpoint_candidates_sha256"
]
assert sha256(threshold_path) == receipt["validation_artifacts"][
    "threshold_candidates_sha256"
]
assert sha256(threshold_path) == receipt["threshold"][
    "candidate_artifact_sha256"
]

git_status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

print(json.dumps(receipt, indent=2, sort_keys=True))
print("\nRECEIPT SHA256:", sha256(receipt_path))
print("GIT STATUS:", repr(git_status))
print("SELECTION RECEIPT VALIDATION: PASS")

files.download(str(receipt_path))

{
  "checkpoints": [
    {
      "drive_relative_path": "runs/seed_17/weights/epoch82.pt",
      "epoch": 83,
      "seed": 17,
      "sha256": "793547a5ec31954d8e909b2f5c63f378374134353a8d1e0cefdd452a5365eefa",
      "size_bytes": 16085716,
      "validation_map_50_95": 0.43109
    },
    {
      "drive_relative_path": "runs/seed_29/weights/epoch85.pt",
      "epoch": 86,
      "seed": 29,
      "sha256": "d799f11b13d88c73e6a0473fb33f00943ff0975fbd25ef8202ef7f17f3e14874",
      "size_bytes": 16086164,
      "validation_map_50_95": 0.42675
    },
    {
      "drive_relative_path": "runs/seed_43/weights/epoch80.pt",
      "epoch": 81,
      "seed": 43,
      "sha256": "99be27499d56f2321fc32a0742f457540629dffdab29457e2bf7d3dd4d280cf3",
      "size_bytes": 16085524,
      "validation_map_50_95": 0.40643
    }
  ],
  "configuration": {
    "path": "configs/detection_phase11b.yaml",
    "sha256": "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
  },
  "frozen_hashes": {
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import importlib.metadata
import subprocess
import sys

assert sys.version_info[:2] == (3, 11), (
    f"STOP: Python 3.11 required, observed {sys.version}"
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--requirement",
        f"{REPOSITORY_ROOT}/requirements-detection-colab.txt",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--editable",
        REPOSITORY_ROOT,
    ],
    check=True,
)

import windblade

print("PYTHON:", sys.version.split()[0])
print("TORCH:", importlib.metadata.version("torch"))
print("TORCHVISION:", importlib.metadata.version("torchvision"))
print("ULTRALYTICS:", importlib.metadata.version("ultralytics"))
print("WINDBLADE IMPORT:", windblade.__file__)
print("ENVIRONMENT INSTALLATION: PASS")

ModuleNotFoundError: No module named 'windblade'

In [12]:
from pathlib import Path
import hashlib
import subprocess

COMMITTED_RECEIPT_SHA = (
    "6c236e9d7220b443f17a628a3d8f621afc56be3777949eeca47f462879e46509"
)
FINAL_SELECTION_COMMIT = "2b1a82f0f73ef86c8b6cd911d41f5eacd48cc8bf"

root = Path(REPOSITORY_ROOT)
receipt = root / "provenance" / "phase11b_selection_receipt.json"
backup = Path("/content/phase11b_selection_receipt.generated.json")

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

print("OLD HEAD:", git("rev-parse", "HEAD"))
print("OLD STATUS:", repr(git("status", "--porcelain")))
print("GENERATED RECEIPT SHA:", sha256(receipt))

assert sha256(receipt) == COMMITTED_RECEIPT_SHA
assert not backup.exists(), f"STOP: backup already exists: {backup}"

# Preserve the generated receipt outside the repository before checkout.
receipt.rename(backup)
assert git("status", "--porcelain") == ""

git("fetch", "origin", "main")
git("cat-file", "-e", FINAL_SELECTION_COMMIT + "^{commit}")
git("checkout", "--detach", FINAL_SELECTION_COMMIT)

assert receipt.is_file()
assert sha256(receipt) == COMMITTED_RECEIPT_SHA
assert receipt.read_bytes() == backup.read_bytes()
assert git("status", "--porcelain") == ""
git("ls-files", "--error-unmatch", "provenance/phase11b_selection_receipt.json")

print("NEW HEAD:", git("rev-parse", "HEAD"))
print("COMMITTED RECEIPT SHA:", sha256(receipt))
print("RECEIPT BYTES MATCH GENERATED COPY:", receipt.read_bytes() == backup.read_bytes())
print("FINAL STATUS:", repr(git("status", "--porcelain")))
print("COMMITTED SELECTION CHECKOUT: PASS")

OLD HEAD: 764dd67b31616fa0165a1fe4058b82eb978e4178
OLD STATUS: '?? provenance/phase11b_selection_receipt.json'
GENERATED RECEIPT SHA: 6c236e9d7220b443f17a628a3d8f621afc56be3777949eeca47f462879e46509
NEW HEAD: 2b1a82f0f73ef86c8b6cd911d41f5eacd48cc8bf
COMMITTED RECEIPT SHA: 6c236e9d7220b443f17a628a3d8f621afc56be3777949eeca47f462879e46509
RECEIPT BYTES MATCH GENERATED COPY: True
FINAL STATUS: ''
COMMITTED SELECTION CHECKOUT: PASS


In [13]:
import subprocess
import sys

validation_code = r"""
from pathlib import Path
import json
import sys

from windblade.detection.phase11b import (
    load_apparatus,
    validate_receipt_firewall,
)

repo = Path(sys.argv[1])
drive_root = Path(sys.argv[2])
config_path = repo / "configs" / "detection_phase11b.yaml"
config = load_apparatus(config_path)

receipt = validate_receipt_firewall(
    config,
    config_path,
    repo,
    drive_root,
)

summary = {
    "status": receipt["status"],
    "test_evaluated": receipt["test_evaluated"],
    "seeds": receipt["seeds"],
    "selected_epochs": {
        str(item["seed"]): item["epoch"]
        for item in receipt["checkpoints"]
    },
    "threshold": receipt["threshold"]["value"],
    "validation_f1": receipt["threshold"]["validation_f1"],
}

print(json.dumps(summary, indent=2, sort_keys=True))
print("COMMITTED RECEIPT FIREWALL: PASS")
"""

result = subprocess.run(
    [
        sys.executable,
        "-c",
        validation_code,
        REPOSITORY_ROOT,
        DRIVE_ROOT,
    ],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

assert result.returncode == 0, "Receipt firewall failed; do not continue"

RETURN CODE: 0
STDOUT:
{
  "seeds": [
    17,
    29,
    43
  ],
  "selected_epochs": {
    "17": 83,
    "29": 86,
    "43": 81
  },
  "status": "FROZEN_BEFORE_TEST",
  "test_evaluated": false,
  "threshold": 0.39,
  "validation_f1": 0.6567967698519516
}
COMMITTED RECEIPT FIREWALL: PASS

STDERR:



In [14]:
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

subprocess.run(
    BASE + ["bundle"],
    cwd=REPOSITORY_ROOT,
    check=True,
)

print("PHASE 11B TRAINING BUNDLE: COMPLETE")

PHASE 11B TRAINING BUNDLE: COMPLETE


In [16]:
from pathlib import Path
import os
import subprocess

final_output = (
    Path(DRIVE_ROOT)
    / "selection"
    / "final_test_metrics.json"
)

print("EXISTING FINAL-TEST OUTPUT:", final_output.exists())
assert not final_output.exists(), (
    "STOP: final_test_metrics.json already exists; final test must not be rerun"
)

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

environment = dict(
    os.environ,
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

subprocess.run(
    BASE + ["final-test"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    check=True,
)

assert final_output.is_file()

print("FINAL-TEST OUTPUT:", final_output)
print("PHASE 11B FINAL TEST: COMPLETE")
print("NO FURTHER THRESHOLD, CHECKPOINT, OR MODEL TUNING IS PERMITTED")

EXISTING FINAL-TEST OUTPUT: False


CalledProcessError: Command '['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'final-test']' returned non-zero exit status 1.

In [17]:
from pathlib import Path
import json
import subprocess

drive_root = Path(DRIVE_ROOT)
test_root = Path(DATA_ROOT).with_name(Path(DATA_ROOT).name + "_test")
final_output = drive_root / "selection" / "final_test_metrics.json"
materialization = test_root / "materialization.json"

print("FINAL OUTPUT EXISTS:", final_output.is_file())
print("TEST ROOT EXISTS:", test_root.exists())
print("TEST MATERIALIZATION EXISTS:", materialization.is_file())

if materialization.is_file():
    record = json.loads(materialization.read_text())
    print("TEST MATERIALIZATION SUMMARY:", {
        "status": record.get("status"),
        "scope": record.get("scope"),
        "included_splits": record.get("included_splits"),
        "image_count": record.get("image_count"),
        "box_count": record.get("box_count"),
        "artifact_fingerprint": record.get("artifact_fingerprint"),
    })

test_images = test_root / "dataset" / "images" / "test"
test_labels = test_root / "dataset" / "labels" / "test"

print(
    "MATERIALIZED TEST IMAGES:",
    len(list(test_images.glob("*"))) if test_images.is_dir() else 0,
)
print(
    "MATERIALIZED TEST LABELS:",
    len(list(test_labels.glob("*.txt"))) if test_labels.is_dir() else 0,
)

selection_files = drive_root / "selection"
print("\nDRIVE SELECTION FILES:")
for path in sorted(selection_files.glob("*")):
    if path.is_file():
        print(path.name, path.stat().st_size)

git_head = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

git_status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

processes = subprocess.run(
    ["ps", "-eo", "pid,etime,args"],
    text=True,
    capture_output=True,
    check=True,
).stdout.splitlines()

active = [
    line for line in processes
    if "run_phase11b.py" in line and "final-test" in line
]

print("\nGIT HEAD:", git_head)
print("GIT STATUS:", repr(git_status))
print("ACTIVE FINAL-TEST PROCESS:", active if active else "NONE")
print("READ-ONLY FAILURE AUDIT: COMPLETE")

FINAL OUTPUT EXISTS: False
TEST ROOT EXISTS: True
TEST MATERIALIZATION EXISTS: True
TEST MATERIALIZATION SUMMARY: {'status': 'PASS', 'scope': 'test', 'included_splits': ['test'], 'image_count': 109, 'box_count': 162, 'artifact_fingerprint': 'be80ee8907fdcd491be1cc6727b810da1a2a4cdd8e3fff0c35b60ac276bc2135'}
MATERIALIZED TEST IMAGES: 109
MATERIALIZED TEST LABELS: 109

DRIVE SELECTION FILES:
validation_checkpoint_candidates.json 82456
validation_threshold_candidates.json 20333

GIT HEAD: 2b1a82f0f73ef86c8b6cd911d41f5eacd48cc8bf
GIT STATUS: ''
ACTIVE FINAL-TEST PROCESS: NONE
READ-ONLY FAILURE AUDIT: COMPLETE


In [18]:
from pathlib import Path
import subprocess
import sys

test_yaml = (
    Path(DATA_ROOT).with_name(Path(DATA_ROOT).name + "_test")
    / "dataset"
    / "test.yaml"
)

print("===== GENERATED TEST YAML =====")
print(test_yaml.read_text())

diagnostic_code = r"""
import sys
import traceback
from ultralytics.data.utils import check_det_dataset

try:
    check_det_dataset(sys.argv[1], autodownload=False)
    print("ULTRALYTICS DATASET CHECK: PASS")
except Exception:
    print("ULTRALYTICS DATASET CHECK: FAILED")
    traceback.print_exc()
    raise SystemExit(1)
"""

result = subprocess.run(
    [sys.executable, "-c", diagnostic_code, str(test_yaml)],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

===== GENERATED TEST YAML =====
path: /content/windblade_phase11b_data_test/dataset
names:
  0: defect
test: images/test

RETURN CODE: 1
STDOUT:
ULTRALYTICS DATASET CHECK: FAILED

STDERR:
Traceback (most recent call last):
  File "<string>", line 7, in <module>
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/data/utils.py", line 417, in check_det_dataset
    raise SyntaxError(
SyntaxError: /content/windblade_phase11b_data_test/dataset/test.yaml 'train:' key missing ❌.
'train' and 'val' are required in all data YAMLs.



In [19]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

CORRECTION_COMMIT = "3f6bdc3916446d90b6a691d84bda6f24c7f90ffa"
EXPECTED_RECEIPT_SHA = (
    "6c236e9d7220b443f17a628a3d8f621afc56be3777949eeca47f462879e46509"
)
EXPECTED_CONFIG_SHA = (
    "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
)
EXPECTED_TEST_FINGERPRINT = (
    "be80ee8907fdcd491be1cc6727b810da1a2a4cdd8e3fff0c35b60ac276bc2135"
)

root = Path(REPOSITORY_ROOT)
receipt_path = root / "provenance" / "phase11b_selection_receipt.json"
failure_path = root / "provenance" / "phase11b_final_test_attempt_1.json"
config_path = root / "configs" / "detection_phase11b.yaml"
final_output = Path(DRIVE_ROOT) / "selection" / "final_test_metrics.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

assert git("status", "--porcelain") == ""

git("fetch", "origin", "main")
git("cat-file", "-e", CORRECTION_COMMIT + "^{commit}")
git("checkout", "--detach", CORRECTION_COMMIT)

failure = json.loads(failure_path.read_text())

assert git("rev-parse", "HEAD") == CORRECTION_COMMIT
assert git("status", "--porcelain") == ""
assert sha256(receipt_path) == EXPECTED_RECEIPT_SHA
assert sha256(config_path) == EXPECTED_CONFIG_SHA
assert failure["status"] == "FAILED_BEFORE_INFERENCE"
assert failure["attempt_number"] == 1
assert failure["final_metrics_written"] is False
assert failure["predictions_generated"] is False
assert failure["selection_changed"] is False
assert failure["tuning_performed"] is False
assert failure["test_materialization_fingerprint"] == EXPECTED_TEST_FINGERPRINT
assert not final_output.exists()

firewall_code = r"""
from pathlib import Path
import sys
from windblade.detection.phase11b import load_apparatus, validate_receipt_firewall

repo = Path(sys.argv[1])
drive_root = Path(sys.argv[2])
config_path = repo / "configs/detection_phase11b.yaml"
config = load_apparatus(config_path)
validate_receipt_firewall(config, config_path, repo, drive_root)
print("COMMITTED RECEIPT FIREWALL: PASS")
"""

result = subprocess.run(
    [
        sys.executable,
        "-c",
        firewall_code,
        REPOSITORY_ROOT,
        DRIVE_ROOT,
    ],
    text=True,
    capture_output=True,
)

print("HEAD:", git("rev-parse", "HEAD"))
print("RECEIPT SHA:", sha256(receipt_path))
print("CONFIG SHA:", sha256(config_path))
print("FAILURE RECORD STATUS:", failure["status"])
print("FINAL OUTPUT EXISTS:", final_output.exists())
print("FIREWALL RETURN CODE:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0
print("FINAL-TEST RETRY GATE: PASS")

HEAD: 3f6bdc3916446d90b6a691d84bda6f24c7f90ffa
RECEIPT SHA: 6c236e9d7220b443f17a628a3d8f621afc56be3777949eeca47f462879e46509
CONFIG SHA: fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b
FAILURE RECORD STATUS: FAILED_BEFORE_INFERENCE
FINAL OUTPUT EXISTS: False
FIREWALL RETURN CODE: 0
COMMITTED RECEIPT FIREWALL: PASS


FINAL-TEST RETRY GATE: PASS


In [20]:
from pathlib import Path
import os
import subprocess

final_output = (
    Path(DRIVE_ROOT)
    / "selection"
    / "final_test_metrics.json"
)

assert not final_output.exists(), (
    "STOP: final_test_metrics.json already exists"
)

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

environment = dict(
    os.environ,
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

result = subprocess.run(
    BASE + ["final-test"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)
print("FINAL OUTPUT EXISTS:", final_output.is_file())

if result.returncode != 0:
    raise RuntimeError(
        "Corrected final evaluation failed. Do not rerun it."
    )

assert final_output.is_file()
print("PHASE 11B FINAL TEST: COMPLETE")
print("NO FURTHER MODEL, CHECKPOINT, THRESHOLD, OR NMS TUNING")

RETURN CODE: 0
STDOUT:
Ultralytics 8.3.150 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2271.8±668.9 MB/s, size: 128.6 KB)
val: New cache created: /content/windblade_phase11b_data_test/dataset/labels/test.cache
                   all        109        162      0.653      0.623      0.659      0.324
Speed: 0.2ms preprocess, 5.6ms inference, 0.0ms loss, 3.7ms postprocess per image
Ultralytics 8.3.150 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1596.9±618.5 MB/s, size: 67.5 KB)
                   all        109        162      0.744       0.61      0.681      0.334
Speed: 0.3ms preprocess, 3.8ms inference, 0.0ms loss, 3.4ms postprocess per image
Ultralytics 8.3.150 🚀 Python-3.11.13 torch-2.6

In [21]:
from pathlib import Path
import hashlib
import json
import math
import statistics

from google.colab import files

final_path = (
    Path(DRIVE_ROOT)
    / "selection"
    / "final_test_metrics.json"
)
receipt_path = (
    Path(REPOSITORY_ROOT)
    / "provenance"
    / "phase11b_selection_receipt.json"
)

final = json.loads(final_path.read_text())
receipt = json.loads(receipt_path.read_text())

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

assert final["status"] == "FINAL_TEST_COMPLETE_NO_FURTHER_TUNING"
assert final["no_post_test_tuning"] is True
assert final["receipt_sha256"] == sha256(receipt_path)
assert final["configuration_sha256"] == receipt["configuration"]["sha256"]
assert [row["seed"] for row in final["per_seed"]] == [17, 29, 43]

receipt_checkpoints = {
    row["seed"]: row["sha256"]
    for row in receipt["checkpoints"]
}

for row in final["per_seed"]:
    assert row["checkpoint_sha256"] == receipt_checkpoints[row["seed"]]
    assert row["frozen_operating_threshold"] == 0.39

values = [row["map_50_95"] for row in final["per_seed"]]
calculated_mean = statistics.mean(values)
calculated_sd = statistics.stdev(values)

assert math.isclose(
    calculated_mean,
    final["aggregate_map_50_95"]["mean"],
    rel_tol=0,
    abs_tol=1e-15,
)
assert math.isclose(
    calculated_sd,
    final["aggregate_map_50_95"]["sample_sd"],
    rel_tol=0,
    abs_tol=1e-15,
)

print(json.dumps(final, indent=2, sort_keys=True))
print("\nFINAL METRICS SHA256:", sha256(final_path))
print("FINAL METRICS VALIDATION: PASS")
print("NO POST-TEST TUNING: ENFORCED")

files.download(str(final_path))

{
  "aggregate_map_50_95": {
    "mean": 0.32600764819381234,
    "sample_sd": 0.007152849550503072
  },
  "configuration_sha256": "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b",
  "no_post_test_tuning": true,
  "per_seed": [
    {
      "checkpoint_sha256": "793547a5ec31954d8e909b2f5c63f378374134353a8d1e0cefdd452a5365eefa",
      "frozen_operating_threshold": 0.39,
      "frozen_threshold_f1": 0.5692883895131086,
      "frozen_threshold_false_negative": 86,
      "frozen_threshold_false_positive": 29,
      "frozen_threshold_true_positive": 76,
      "map_50": 0.6586445290736995,
      "map_50_95": 0.3243185659488687,
      "precision": 0.6530641133692915,
      "recall": 0.6234567901234568,
      "seed": 17
    },
    {
      "checkpoint_sha256": "d799f11b13d88c73e6a0473fb33f00943ff0975fbd25ef8202ef7f17f3e14874",
      "frozen_operating_threshold": 0.39,
      "frozen_threshold_f1": 0.5682656826568266,
      "frozen_threshold_false_negative": 85,
      "frozen_thr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sys

EXPECTED_PYTHON = (3, 11)

if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "Phase 11B requires Colab runtime version 2025.07 "
        "(Python 3.11). Select Runtime > Change runtime type > "
        "Runtime version: 2025.07, choose T4 GPU, reconnect, "
        "and restart from this cell."
    )

print("Python version gate passed:", sys.version)

Python version gate passed: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/Stukhori/Bade-defect-recognition.git'
APPARATUS_COMMIT = 'e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272'
REPOSITORY_ROOT = '/content/Bade-defect-recognition'
DRIVE_ROOT = '/content/drive/MyDrive/windblade_phase11b'
DATA_ROOT = '/content/windblade_phase11b_data'

Mounted at /content/drive


In [ ]:
import pathlib, shutil, subprocess
if APPARATUS_COMMIT == 'REPLACE_WITH_APPARATUS_COMMIT':
    raise ValueError('Set the full apparatus commit before continuing')
if pathlib.Path(REPOSITORY_ROOT).exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, REPOSITORY_ROOT], check=True)
subprocess.run(['git', 'checkout', '--detach', APPARATUS_COMMIT], cwd=REPOSITORY_ROOT, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_ROOT, text=True).strip()
assert head == APPARATUS_COMMIT

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '--requirement', f'{REPOSITORY_ROOT}/requirements-detection-colab.txt'], check=True)

CompletedProcess(args=['python', '-m', 'pip', 'install', '--requirement', '/content/Bade-defect-recognition/requirements-detection-colab.txt'], returncode=0)

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '--no-deps', '--editable', REPOSITORY_ROOT], check=True)

CompletedProcess(args=['python', '-m', 'pip', 'install', '--no-deps', '--editable', '/content/Bade-defect-recognition'], returncode=0)

In [ ]:
BASE = ['python', 'scripts/run_phase11b.py', '--drive-root', DRIVE_ROOT, '--data-root', DATA_ROOT]
subprocess.run(BASE + ['apparatus-check'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['verify-archive'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['preflight'], cwd=REPOSITORY_ROOT, check=True)

CompletedProcess(args=['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'preflight'], returncode=0)

In [ ]:
record = pathlib.Path(DRIVE_ROOT) / 'provenance/phase11b_weight_acquisition.json'
if not record.exists():
    subprocess.run(BASE + ['acquire-weight', '--apparatus-commit', APPARATUS_COMMIT], cwd=REPOSITORY_ROOT, check=True)
else:
    print('Using the existing immutable weight-acquisition record; training will verify its bytes.')

Using the existing immutable weight-acquisition record; training will verify its bytes.


In [ ]:
subprocess.run(BASE + ['materialize-trainval'], cwd=REPOSITORY_ROOT, check=True)

CompletedProcess(args=['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'materialize-trainval'], returncode=0)

In [ ]:
from pathlib import Path

drive = Path(DRIVE_ROOT)
run = drive / "runs" / "seed_17"

print("SEED 17 DIRECTORY EXISTS:", run.exists())
print("SEED 29 DIRECTORY EXISTS:", (drive / "runs" / "seed_29").exists())
print("SEED 43 DIRECTORY EXISTS:", (drive / "runs" / "seed_43").exists())

print(
    "WEIGHT RECORD EXISTS:",
    (drive / "provenance" / "phase11b_weight_acquisition.json").is_file(),
)
print(
    "INITIAL WEIGHT EXISTS:",
    (drive / "provenance" / "yolo11n.pt").is_file(),
)
print(
    "MATERIALIZATION RECORD EXISTS:",
    (Path(DATA_ROOT) / "materialization.json").is_file(),
)

if run.exists():
    print("\nSEED 17 FILES:")
    for path in sorted(run.rglob("*")):
        if path.is_file():
            print(path.relative_to(run), "SIZE:", path.stat().st_size)

    for relative in ("run_state.json", "args.yaml", "results.csv"):
        path = run / relative
        if path.is_file():
            print(f"\n===== {relative} =====")
            print(path.read_text(encoding="utf-8", errors="replace"))

SEED 17 DIRECTORY EXISTS: True
SEED 29 DIRECTORY EXISTS: True
SEED 43 DIRECTORY EXISTS: False
WEIGHT RECORD EXISTS: True
INITIAL WEIGHT EXISTS: True
MATERIALIZATION RECORD EXISTS: True

SEED 17 FILES:
F1_curve.png SIZE: 112823
PR_curve.png SIZE: 87298
P_curve.png SIZE: 102073
R_curve.png SIZE: 108866
args.yaml SIZE: 1747
confusion_matrix.png SIZE: 89280
confusion_matrix_normalized.png SIZE: 93875
labels.jpg SIZE: 222451
labels_correlogram.jpg SIZE: 241965
results.csv SIZE: 12707
results.png SIZE: 300698
run_state.json SIZE: 508
train_batch0.jpg SIZE: 356189
train_batch1.jpg SIZE: 356807
train_batch2.jpg SIZE: 361789
train_batch2880.jpg SIZE: 308324
train_batch2881.jpg SIZE: 298376
train_batch2882.jpg SIZE: 300942
val_batch0_labels.jpg SIZE: 407013
val_batch0_pred.jpg SIZE: 417252
val_batch1_labels.jpg SIZE: 377699
val_batch1_pred.jpg SIZE: 416725
val_batch2_labels.jpg SIZE: 335599
val_batch2_pred.jpg SIZE: 342206
weights/best.pt SIZE: 5477331
weights/epoch0.pt SIZE: 16075540
weights/ep

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import yaml

runs_root = Path(DRIVE_ROOT) / "runs"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for seed in (17, 29, 43):
    run_dir = runs_root / f"seed_{seed}"
    state_path = run_dir / "run_state.json"
    results_path = run_dir / "results.csv"
    args_path = run_dir / "args.yaml"
    last_path = run_dir / "weights" / "last.pt"

    print(f"\n========== SEED {seed} ==========")
    print("RUN DIRECTORY:", run_dir.exists())

    if state_path.is_file():
        state = json.loads(state_path.read_text())
        print("STATE:", {
            key: state.get(key)
            for key in (
                "seed",
                "status",
                "configuration_sha256",
                "apparatus_commit",
                "applied_batch_size",
                "applied_optimizer",
                "applied_amp",
                "last_checkpoint_sha256",
            )
        })
    else:
        state = {}
        print("STATE: MISSING")

    if results_path.is_file():
        with results_path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        print("RESULT ROWS:", len(rows))
        print("LAST RECORDED EPOCH:", rows[-1].get("epoch") if rows else None)
    else:
        print("RESULTS: MISSING")

    if args_path.is_file():
        args = yaml.safe_load(args_path.read_text())
        print("TRAINING ARGS:", {
            key: args.get(key)
            for key in (
                "data",
                "epochs",
                "batch",
                "optimizer",
                "lr0",
                "cos_lr",
                "seed",
                "resume",
            )
        })
    else:
        print("ARGS: MISSING")

    epoch_files = list((run_dir / "weights").glob("epoch*.pt"))
    print("EPOCH CHECKPOINT COUNT:", len(epoch_files))
    print("LAST.PT EXISTS:", last_path.is_file())

    if last_path.is_file():
        observed_sha = sha256(last_path)
        print("LAST.PT SHA256:", observed_sha)
        print(
            "STATE HASH MATCH:",
            state.get("last_checkpoint_sha256") == observed_sha
            if state.get("last_checkpoint_sha256")
            else "NO HASH RECORDED"
        )


========== SEED 17 ==========
RUN DIRECTORY: True
STATE: {'seed': 17, 'status': None, 'configuration_sha256': 'fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b', 'apparatus_commit': 'e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272', 'applied_batch_size': None, 'applied_optimizer': None, 'applied_amp': None, 'last_checkpoint_sha256': None}
RESULT ROWS: 100
LAST RECORDED EPOCH: 100
TRAINING ARGS: {'data': '/content/windblade_phase11b_data/dataset/trainval.yaml', 'epochs': 100, 'batch': 16, 'optimizer': 'AdamW', 'lr0': 0.001, 'cos_lr': True, 'seed': 17, 'resume': '/content/drive/MyDrive/windblade_phase11b/runs/seed_17/weights/last.pt'}
EPOCH CHECKPOINT COUNT: 100
LAST.PT EXISTS: True
LAST.PT SHA256: b588d29a3d03ad4315ed7d6a3fb8c8c0875eaf193b2614e1b2972987b1b2cb99
STATE HASH MATCH: NO HASH RECORDED

========== SEED 29 ==========
RUN DIRECTORY: True
STATE: {'seed': 29, 'status': None, 'configuration_sha256': 'fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b', 'appa

In [ ]:
from pathlib import Path
import hashlib
import subprocess

RECOVERY_COMMIT = "749cc29f81b5223edd36dddc3928b1814d029096"
EXPECTED_CONFIG_SHA256 = (
    "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
)

root = Path(REPOSITORY_ROOT)
config_path = root / "configs" / "detection_phase11b.yaml"

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

initial_status = git("status", "--porcelain")
print("OLD HEAD:", git("rev-parse", "HEAD"))
print("INITIAL STATUS:", repr(initial_status))

assert initial_status == "", "STOP: repository is not clean"

git("fetch", "origin", "main")
git("cat-file", "-e", RECOVERY_COMMIT + "^{commit}")
git("checkout", "--detach", RECOVERY_COMMIT)

config_sha256 = hashlib.sha256(config_path.read_bytes()).hexdigest()
final_status = git("status", "--porcelain")

print("NEW HEAD:", git("rev-parse", "HEAD"))
print("CONFIG SHA256:", config_sha256)
print("FINAL STATUS:", repr(final_status))

assert git("rev-parse", "HEAD") == RECOVERY_COMMIT
assert config_sha256 == EXPECTED_CONFIG_SHA256
assert final_status == ""

print("RECOVERY COMMIT CHECKOUT: PASS")

OLD HEAD: e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272
INITIAL STATUS: ''
NEW HEAD: 749cc29f81b5223edd36dddc3928b1814d029096
CONFIG SHA256: fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b
FINAL STATUS: ''
RECOVERY COMMIT CHECKOUT: PASS


In [ ]:
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

for command in ("apparatus-check", "verify-archive", "preflight"):
    result = subprocess.run(
        BASE + [command],
        cwd=REPOSITORY_ROOT,
        text=True,
        capture_output=True,
    )

    print(f"\n========== {command} ==========")
    print("RETURN CODE:", result.returncode)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"{command} failed; stop here")

print("\nALL PRE-TRAINING CHECKS: PASS")


========== apparatus-check ==========
RETURN CODE: 0
STDOUT:
{
  "boxes": 1065,
  "boxes_by_split": {
    "test": 162,
    "train": 757,
    "validation": 146
  },
  "config_fingerprint": "9f4a20ba4404c9a6072277a504c466a0756143b908e79c7168d2ccf91ff32057",
  "dataset_fingerprint": "ad4ab59c3e3c85c6cf0b85b148177bd6b79d24f372f49bdff0043609e6fefc97",
  "images": 720,
  "images_by_split": {
    "test": 109,
    "train": 510,
    "validation": 101
  },
  "run_count": 3,
  "scientific_output_fingerprint": "3f46cbdc6c7a2e3cf6093ff177dd1948d113fa4c36fa9eb907d7c8621e800461",
  "seeds": [
    17,
    29,
    43
  ],
  "split_fingerprint": "264f8460f203074374c2c098c8fd5d2e55fb7ee1f281a8d505e2dfb0de9a2bc3",
  "status": "PASS"
}

STDERR:


========== verify-archive ==========
RETURN CODE: 0
STDOUT:
{
  "filename": "WT blade defect dataset.zip",
  "sha256": "466452f2a0cfc9ef6ba63ea2a3bbc7ea4262057dd07e4fc9e00eedf5bba305b4",
  "size_bytes": 78958553,
  "status": "PASS"
}

STDERR:


========== preflig

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess

RECOVERY_COMMIT = "749cc29f81b5223edd36dddc3928b1814d029096"
run_dir = Path(DRIVE_ROOT) / "runs" / "seed_17"
weights_dir = run_dir / "weights"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protected_files = [
    run_dir / "results.csv",
    run_dir / "args.yaml",
    weights_dir / "best.pt",
    weights_dir / "last.pt",
]

before_hashes = {str(path): sha256(path) for path in protected_files}
before_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

result = subprocess.run(
    BASE + ["train", "--seed", "17"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Seed 17 recovery failed; stop here")

after_hashes = {str(path): sha256(path) for path in protected_files}
after_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

state = json.loads((run_dir / "run_state.json").read_text())

print("STATUS:", state.get("status"))
print("LAST CHECKPOINT SHA256:", state.get("last_checkpoint_sha256"))
print("APPLIED BATCH:", state.get("applied_batch_size"))
print("APPLIED OPTIMIZER:", state.get("applied_optimizer"))
print("APPLIED AMP:", state.get("applied_amp"))
print("RECOVERY:", state.get("completion_recovery"))
print("PROTECTED HASHES UNCHANGED:", before_hashes == after_hashes)
print(
    "EPOCH CHECKPOINTS UNCHANGED:",
    before_epoch_metadata == after_epoch_metadata,
)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["last_checkpoint_sha256"] == sha256(weights_dir / "last.pt")
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["completion_recovery"]["training_invoked"] is False
assert state["completion_recovery"]["repository_commit"] == RECOVERY_COMMIT
assert before_hashes == after_hashes
assert before_epoch_metadata == after_epoch_metadata

print("SEED 17 SAFE RECOVERY: PASS")

RETURN CODE: 1
STDOUT:

STDERR:
Traceback (most recent call last):
  File "/content/Bade-defect-recognition/scripts/run_phase11b.py", line 85, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/Bade-defect-recognition/scripts/run_phase11b.py", line 69, in main
    result = train_seed(config, config_path, repo, data_root, drive_root, weight, weight_record, args.seed)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Bade-defect-recognition/src/windblade/detection/phase11b.py", line 719, in train_seed
    recovered = recover_legacy_completed_run(config, repo, data_root, layout, run_dir, seed, state, expected_state)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Bade-defect-recognition/src/windblade/detection/phase11b.py", line 658, in recover_legacy_completed_run
    applied = _critical_training

RuntimeError: Seed 17 recovery failed; stop here

In [ ]:
from pathlib import Path
import hashlib
import subprocess

NEW_COMMIT = "764dd67b31616fa0165a1fe4058b82eb978e4178"
EXPECTED_CONFIG_SHA256 = (
    "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
)

root = Path(REPOSITORY_ROOT)
config_path = root / "configs" / "detection_phase11b.yaml"

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

print("OLD HEAD:", git("rev-parse", "HEAD"))
print("INITIAL STATUS:", repr(git("status", "--porcelain")))

assert git("status", "--porcelain") == "", "STOP: repository is not clean"

git("fetch", "origin", "main")
git("cat-file", "-e", NEW_COMMIT + "^{commit}")
git("checkout", "--detach", NEW_COMMIT)

config_sha = hashlib.sha256(config_path.read_bytes()).hexdigest()

print("NEW HEAD:", git("rev-parse", "HEAD"))
print("CONFIG SHA256:", config_sha)
print("FINAL STATUS:", repr(git("status", "--porcelain")))

assert git("rev-parse", "HEAD") == NEW_COMMIT
assert config_sha == EXPECTED_CONFIG_SHA256
assert git("status", "--porcelain") == ""

print("DEVICE-NORMALIZATION COMMIT CHECKOUT: PASS")

OLD HEAD: 749cc29f81b5223edd36dddc3928b1814d029096
INITIAL STATUS: ''
NEW HEAD: 764dd67b31616fa0165a1fe4058b82eb978e4178
CONFIG SHA256: fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b
FINAL STATUS: ''
DEVICE-NORMALIZATION COMMIT CHECKOUT: PASS


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess

RECOVERY_COMMIT = "764dd67b31616fa0165a1fe4058b82eb978e4178"
run_dir = Path(DRIVE_ROOT) / "runs" / "seed_17"
weights_dir = run_dir / "weights"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protected_files = [
    run_dir / "results.csv",
    run_dir / "args.yaml",
    weights_dir / "best.pt",
    weights_dir / "last.pt",
]

before_hashes = {str(path): sha256(path) for path in protected_files}
before_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

result = subprocess.run(
    BASE + ["train", "--seed", "17"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Seed 17 recovery failed; stop here")

after_hashes = {str(path): sha256(path) for path in protected_files}
after_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

state = json.loads((run_dir / "run_state.json").read_text())

print("STATUS:", state.get("status"))
print("LAST CHECKPOINT SHA256:", state.get("last_checkpoint_sha256"))
print("APPLIED BATCH:", state.get("applied_batch_size"))
print("APPLIED OPTIMIZER:", state.get("applied_optimizer"))
print("APPLIED AMP:", state.get("applied_amp"))
print("RECOVERY:", state.get("completion_recovery"))
print("PROTECTED HASHES UNCHANGED:", before_hashes == after_hashes)
print("EPOCH CHECKPOINTS UNCHANGED:", before_epoch_metadata == after_epoch_metadata)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["last_checkpoint_sha256"] == sha256(weights_dir / "last.pt")
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["completion_recovery"]["training_invoked"] is False
assert state["completion_recovery"]["repository_commit"] == RECOVERY_COMMIT
assert before_hashes == after_hashes
assert before_epoch_metadata == after_epoch_metadata

print("SEED 17 SAFE RECOVERY: PASS")

RETURN CODE: 0
STDOUT:
{
  "apparatus_commit": "e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272",
  "applied_amp": true,
  "applied_batch_size": 16,
  "applied_optimizer": "AdamW",
  "completion_recovery": {
    "mode": "VERIFIED_PRE_EXISTING_ARTIFACTS",
    "repository_commit": "764dd67b31616fa0165a1fe4058b82eb978e4178",
    "training_invoked": false
  },
  "configuration_path": "configs/detection_phase11b.yaml",
  "configuration_sha256": "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b",
  "last_checkpoint_sha256": "b588d29a3d03ad4315ed7d6a3fb8c8c0875eaf193b2614e1b2972987b1b2cb99",
  "materialization_fingerprint": "dd5ceb31ef32e9f9af2adeeaa9d3ebcf263eb232621d924845ff2a53785fc011",
  "seed": 17,
  "status": "TRAINING_COMMAND_COMPLETED",
  "weight_path": "/content/drive/MyDrive/windblade_phase11b/provenance/yolo11n.pt",
  "weight_sha256": "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1"
}

STDERR:

STATUS: TRAINING_COMMAND_COMPLETED
LAST CHECKPOINT SHA256: b

In [ ]:
import os
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

environment = dict(
    os.environ,
    PYTHONHASHSEED="29",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

subprocess.run(
    BASE + ["train", "--seed", "29"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    check=True,
)

CompletedProcess(args=['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'train', '--seed', '29'], returncode=0)

In [ ]:
from pathlib import Path
import csv
import hashlib
import json

run_dir = Path(DRIVE_ROOT) / "runs" / "seed_29"
state_path = run_dir / "run_state.json"
results_path = run_dir / "results.csv"
last_path = run_dir / "weights" / "last.pt"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

state = json.loads(state_path.read_text())

with results_path.open(newline="") as handle:
    rows = list(csv.DictReader(handle))

observed_last_sha = sha256(last_path)

print("STATUS:", state.get("status"))
print("RESULT ROWS:", len(rows))
print("LAST RECORDED EPOCH:", rows[-1].get("epoch"))
print("APPLIED BATCH:", state.get("applied_batch_size"))
print("APPLIED OPTIMIZER:", state.get("applied_optimizer"))
print("APPLIED AMP:", state.get("applied_amp"))
print("RECORDED LAST SHA:", state.get("last_checkpoint_sha256"))
print("OBSERVED LAST SHA:", observed_last_sha)
print(
    "CHECKPOINT COUNT:",
    len(list((run_dir / "weights").glob("epoch*.pt"))),
)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["seed"] == 29
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["last_checkpoint_sha256"] == observed_last_sha
assert len(rows) > 30

print("SEED 29 COMPLETION: PASS")

STATUS: TRAINING_COMMAND_COMPLETED
RESULT ROWS: 100
LAST RECORDED EPOCH: 100
APPLIED BATCH: 16
APPLIED OPTIMIZER: AdamW
APPLIED AMP: True
RECORDED LAST SHA: f5f0640af079ef5a94404c1d8e17cf81b44f7b287130fc2b85c053199d69f667
OBSERVED LAST SHA: f5f0640af079ef5a94404c1d8e17cf81b44f7b287130fc2b85c053199d69f667
CHECKPOINT COUNT: 100
SEED 29 COMPLETION: PASS


In [ ]:
import os
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

environment = dict(
    os.environ,
    PYTHONHASHSEED="43",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

subprocess.run(
    BASE + ["train", "--seed", "43"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    check=True,
)

In [ ]:
subprocess.run(BASE + ['select-validation'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['bundle'], cwd=REPOSITORY_ROOT, check=True)

NameError: name 'subprocess' is not defined

Stop here. Inspect the generated selection receipt, commit and push it from an authenticated checkout, and follow `docs/phase11b_colab.md` for the separately gated final evaluation. Do not change the frozen configuration or selections.